# Red neuronal: perceptrón multicapa (MLP)
## Proyecto final de Ciencia de Datos · *Bank Marketing* (UCI)

**Instructor:** M. en C. Benito Samuel López Razo · **Autores:** _(escribe aquí tu nombre y el de tu equipo)_ · **Semilla global:** 42

Este notebook vive en la carpeta `modelos/` y es **independiente** de los demás. Todo lo compartido (carga de datos, división, validación cruzada y funciones de evaluación) está en `utils_proyecto.py`, para que todos los modelos se comparen bajo las mismas reglas (las reglas están en la sección 1 de `01_regresion_logistica.ipynb`).

El PDF pide un **perceptrón multicapa con una arquitectura justificada (capas, neuronas, activación) y algún mecanismo para evitar el sobreajuste**, con ajuste de hiperparámetros por validación cruzada y reportando el espacio de búsqueda.

**Índice**

1. Carga de datos
2. Arquitectura, justificación y espacio de búsqueda
3. Búsqueda en malla con validación cruzada
4. Evaluación en el conjunto de prueba
5. Curvas de aprendizaje: ¿se detuvo a tiempo?
6. ¿Funcionan realmente la regularización y la detención temprana?
7. Variabilidad: entre folds y entre semillas
8. Interpretación

## 1. Carga de datos

La celda siguiente importa lo necesario y carga los datos con `utils_proyecto.py`. **La tabla de control debe dar 31,647 / 6,782 / 6,782 filas y 11.70 % de `yes`** en cada conjunto, igual que la sección 6 de la Parte 1.

`N_JOBS = -1` usa todos los núcleos del procesador; si tu computadora se pone lenta, cámbialo a `2`.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

import utils_proyecto as up

SEED = up.SEED
N_JOBS = -1
d = up.cargar_datos()

display(up.resumen_datos(d))

,filas,% clase 'yes'
train,31647,11.70
validation,6782,11.69
test,6782,11.69


## 2. Arquitectura, justificación y espacio de búsqueda

| Decisión | Elección | Justificación |
|---|---|---|
| **Entrada** | Las 48 columnas del preprocesamiento (6 numéricas **estandarizadas**, el resto *one-hot* y una binaria) | El escalado es obligatorio para una red neuronal: sin él, variables como `balance` (en miles) dominarían a las demás |
| **Capas ocultas** | 1 a 3 capas, con 16 a 256 neuronas en la primera | Son datos tabulares de ~31 mil filas y 48 variables: una red grande memorizaría el entrenamiento y no hay estructura espacial o secuencial que justifique una red profunda. Se empieza pequeño y la validación cruzada decide si más capacidad ayuda |
| **Activación oculta** | `relu` o `tanh` (se comparan) | ReLU es el estándar (no se satura con valores grandes); tanh es suave y acotada, y a veces generaliza mejor con pocos datos |
| **Salida** | 1 neurona logística (sigmoide) con pérdida de entropía cruzada | Es clasificación binaria: la salida se interpreta como la probabilidad de `yes` |
| **Optimizador** | Adam, tasa de aprendizaje 0.001, lotes de 200 | Valores por defecto robustos; no se ajustan para mantener la búsqueda manejable |
| **Contra el sobreajuste (1)** | **Regularización L2** (`alpha`) | Penaliza pesos grandes; con `alpha` mayor la red es más simple |
| **Contra el sobreajuste (2)** | **Detención temprana** | Se aparta el 10 % del fold de entrenamiento y se detiene el entrenamiento cuando el puntaje en ese 10 % no mejora durante 10 épocas seguidas (máximo 300 épocas) |
| **Desbalance de clases** | Sin remuestreo; el desbalance se corrige con el **umbral** elegido en validación | `MLPClassifier` no tiene `class_weight` |

**Espacio de búsqueda** (malla):

| Hiperparámetro | Valores |
|---|---|
| `hidden_layer_sizes` | (16,), (32,), (64,), (32, 16), (64, 32), (128, 64), (256, 128), (128, 64, 32) |
| `activation` | `relu`, `tanh` |
| `alpha` (L2) | 0.0001, 0.01, 0.1, 1, 10 |

8 × 2 × 5 = 80 combinaciones × 5 folds = 400 ajustes. La semilla del modelo es `random_state = 42` en todos.

**Nota sobre la detención temprana en scikit-learn:** el 10 % que aparta se usa para medir la **exactitud**, no el AUC-PR. Con clases desbalanceadas la exactitud es una señal tosca; por eso la detención temprana se **complementa con L2** y en la sección 6 se verifica con una métrica adecuada (AUC-PR en *train* contra validación) que realmente controla el sobreajuste.

## 3. Búsqueda en malla con validación cruzada

`GridSearchCV` con validación cruzada estratificada de 5 folds sobre *train* y métrica AUC-PR. El preprocesamiento va dentro del `Pipeline` (se ajusta en cada fold). Al terminar, `GridSearchCV` reentrena la mejor combinación con todo *train*.

In [ ]:
pipe_mlp = Pipeline([
    ("prep", clone(d.preprocesador)),
    ("modelo", MLPClassifier(solver="adam", learning_rate_init=0.001, batch_size=200, max_iter=300,
                             early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
                             random_state=SEED)),
])

espacio_mlp = {
    "modelo__hidden_layer_sizes": [(16,), (32,), (64,), (32, 16), (64, 32), (128, 64), (256, 128), (128, 64, 32)],
    "modelo__activation": ["relu", "tanh"],
    "modelo__alpha": [1e-4, 1e-2, 0.1, 1, 10],
}

busqueda_mlp = GridSearchCV(pipe_mlp, espacio_mlp, scoring=up.METRICA_CV, cv=up.CV, n_jobs=N_JOBS)
busqueda_mlp.fit(d.X_train, d.y_train)

print("Mejores hiperparámetros:", up.formatear_params(busqueda_mlp.best_params_))
print(f"{up.METRICA_CV} medio en CV: {busqueda_mlp.best_score_:.4f}")

res = busqueda_mlp.cv_results_
cv_mlp = pd.DataFrame({
    "capas ocultas": [str(p["modelo__hidden_layer_sizes"]) for p in res["params"]],
    "activación": [p["modelo__activation"] for p in res["params"]],
    "alpha": [p["modelo__alpha"] for p in res["params"]],
    "AUC-PR medio": res["mean_test_score"],
    "desv. estándar": res["std_test_score"],
    "ranking": res["rank_test_score"],
}).sort_values("ranking").reset_index(drop=True)

display(cv_mlp.head(10).round(4))
up.guardar_tabla(cv_mlp.round(4), "tabla_mlp_busqueda_cv")

Mapas de calor del AUC-PR medio en validación cruzada para cada activación (renglones: arquitectura; columnas: `alpha`). Sirven para ver si más neuronas o más regularización realmente ayudan.

In [ ]:
orden_capas = [str(c) for c in espacio_mlp["modelo__hidden_layer_sizes"]]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
vmin, vmax = cv_mlp["AUC-PR medio"].min(), cv_mlp["AUC-PR medio"].max()
for ax, act in zip(axes, ["relu", "tanh"]):
    sub = cv_mlp[cv_mlp["activación"] == act]
    malla = sub.pivot(index="capas ocultas", columns="alpha", values="AUC-PR medio").loc[orden_capas]
    sns.heatmap(malla, annot=True, fmt=".3f", cmap="viridis", vmin=vmin, vmax=vmax,
                cbar=(act == "tanh"), cbar_kws={"label": "AUC-PR medio (CV)"}, ax=ax)
    ax.set_title(f"MLP con activación {act}")
    ax.set_xlabel("alpha (regularización L2)")
    ax.set_ylabel("capas ocultas")
fig.tight_layout()
plt.show()
up.guardar(fig, "fig_mlp_busqueda_cv")

## 4. Evaluación en el conjunto de prueba

`evaluar_modelo()` elige el umbral con el conjunto de **validación**, evalúa **una sola vez** en *test* y genera matriz de confusión, curva ROC y curva precisión-recall. También guarda en `resultados/` las métricas (`mlp_metricas_test.json`) y las predicciones (`mlp_predicciones_test.csv`) para la comparación final y el análisis de errores.

In [ ]:
mejor_mlp = busqueda_mlp.best_estimator_

metricas_mlp = up.evaluar_modelo("Red neuronal (MLP)", mejor_mlp, "mlp", d)
display(metricas_mlp.to_frame("test"))

## 5. Curvas de aprendizaje: ¿se detuvo a tiempo?

Izquierda: la pérdida de entrenamiento baja en cada época. Derecha: la exactitud en el 10 % apartado (validación interna); la línea punteada marca la **mejor época**, y el entrenamiento se detiene 10 épocas después de ella si no hay mejora (detención temprana).

In [ ]:
mlp = mejor_mlp.named_steps["modelo"]
mejor_epoca = int(np.argmax(mlp.validation_scores_)) + 1
print(f"Épocas entrenadas: {mlp.n_iter_} | mejor época según validación interna: {mejor_epoca} | "
      f"mejor exactitud interna: {mlp.best_validation_score_:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(range(1, len(mlp.loss_curve_) + 1), mlp.loss_curve_)
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Pérdida de entrenamiento (entropía cruzada)")
axes[0].set_title("Pérdida de entrenamiento")

axes[1].plot(range(1, len(mlp.validation_scores_) + 1), mlp.validation_scores_)
axes[1].axvline(mejor_epoca, ls="--", color="gray", label=f"mejor época = {mejor_epoca}")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Exactitud en el 10 % apartado")
axes[1].set_title("Validación interna (detención temprana)")
axes[1].legend()

fig.tight_layout()
plt.show()
up.guardar(fig, "fig_mlp_curvas_aprendizaje")

## 6. ¿Funcionan realmente la regularización y la detención temprana?

Para comprobar que los mecanismos contra el sobreajuste hacen algo, se entrena la **misma arquitectura y activación** elegidas en cuatro configuraciones y se compara el AUC-PR en *train* contra el de **validación**. Una **brecha grande** (mucho mejor en *train* que en validación) indica sobreajuste:

1. **Elegida:** L2 con el `alpha` encontrado + detención temprana.
2. **Solo L2:** sin detención temprana (hasta 300 épocas).
3. **Sin ninguna:** `alpha` casi nulo y sin detención temprana.
4. **Solo L2 fuerte:** `alpha = 1` y sin detención temprana (para ver si una regularización más fuerte basta por sí sola).

In [ ]:
configuraciones = {
    "1. Elegida (L2 + detención temprana)": {},
    "2. Solo L2 (sin detención temprana)": {"modelo__early_stopping": False},
    "3. Sin ninguna (alpha ≈ 0, sin detención temprana)": {"modelo__early_stopping": False, "modelo__alpha": 1e-8},
    "4. Solo L2 fuerte (alpha = 1, sin detención temprana)": {"modelo__early_stopping": False, "modelo__alpha": 1.0},
}

filas = []
for nombre, cambios in configuraciones.items():
    m = clone(mejor_mlp).set_params(**cambios).fit(d.X_train, d.y_train)
    ap_train = average_precision_score(d.y_train, m.predict_proba(d.X_train)[:, 1])
    ap_val = average_precision_score(d.y_val, m.predict_proba(d.X_val)[:, 1])
    filas.append({"configuración": nombre, "épocas": m.named_steps["modelo"].n_iter_,
                  "AUC-PR train": ap_train, "AUC-PR validación": ap_val, "brecha": ap_train - ap_val})

tabla_sobreajuste = pd.DataFrame(filas)
display(tabla_sobreajuste.round(4))
up.guardar_tabla(tabla_sobreajuste.round(4), "tabla_mlp_sobreajuste")

## 7. Variabilidad: entre folds y entre semillas

El PDF pide la variabilidad "entre folds **o varias semillas**". Una red neuronal depende además de la inicialización aleatoria de sus pesos, así que se miden **las dos**:

- **Entre folds:** validación cruzada repetida 5 × 3 sobre *train* (se guarda en `resultados/mlp_cv_scores.csv` para comparar modelos por pares).
- **Entre semillas:** se reentrena la mejor configuración con 10 semillas distintas y se mide el AUC-PR en **validación** (el conjunto de *test* no se toca).

In [ ]:
tabla_var_mlp = up.variabilidad_cv("Red neuronal (MLP)", mejor_mlp, "mlp", d, n_jobs=N_JOBS)
display(tabla_var_mlp)

In [ ]:
filas = []
for semilla in range(10):
    m = clone(mejor_mlp).set_params(modelo__random_state=semilla).fit(d.X_train, d.y_train)
    p_val = m.predict_proba(d.X_val)[:, 1]
    filas.append({"semilla": semilla, "épocas": m.named_steps["modelo"].n_iter_,
                  "AUC-PR (val)": average_precision_score(d.y_val, p_val),
                  "AUC-ROC (val)": roc_auc_score(d.y_val, p_val)})

por_semilla = pd.DataFrame(filas)
display(por_semilla.round(4))

resumen_semillas = pd.DataFrame({"media": por_semilla[["AUC-PR (val)", "AUC-ROC (val)"]].mean(),
                                 "desv. estándar": por_semilla[["AUC-PR (val)", "AUC-ROC (val)"]].std(ddof=0)}).round(4)
display(resumen_semillas)
up.guardar_tabla(por_semilla.round(4), "tabla_mlp_semillas")

## 8. Interpretación

_Los números de abajo son los de la corrida con semilla 42 y los datos de la Parte 1. Si tu tabla da valores distintos, actualiza el texto con los tuyos._

**Espacio de búsqueda y mejor combinación.** Se probaron 80 combinaciones (8 arquitecturas × 2 activaciones × 5 valores de `alpha`; 400 ajustes con validación cruzada de 5 folds). La mejor fue **dos capas de 128 y 64 neuronas, activación `relu` y `alpha = 0.01`** (AUC-PR en CV 0.430 ± 0.013). Se amplió la malla con (256, 128) y (128, 64, 32) para comprobar que el óptimo no estuviera en el borde: ninguna superó a (128, 64), así que el óptimo queda dentro del espacio de búsqueda.

**Qué importa y qué no.**
- **Capacidad:** una capa de 16 neuronas llega a 0.415; subir a 32 da 0.425, a dos capas (64, 32) 0.428 y a (128, 64) 0.430. Después se estanca ((256, 128): 0.430; tres capas: 0.425). Más neuronas ayudan poco y la ganancia total (≈ 0.015) es del tamaño de una desviación estándar entre folds. Con 48 variables tabulares no hay estructura que justifique una red mayor.
- **Activación:** `relu` supera a `tanh` en las 8 arquitecturas (por 0.005 a 0.013).
- **`alpha`:** entre 0.0001 y 0.1 casi no cambia el resultado; con `alpha = 1` baja ≈ 0.015 y con `alpha = 10` el modelo **subajusta** (0.28–0.32). Que un `alpha` pequeño no dañe probablemente se debe a que la detención temprana ya regulariza la red.
- Los cinco primeros lugares están a menos de 0.003 entre sí, mucho menos que la desviación entre folds (≈ 0.015), así que elegir entre ellos es casi un empate; se conserva el primero por regla.

**Curvas de aprendizaje (sección 5).** La pérdida de entrenamiento baja en cada época, pero la exactitud en el 10 % apartado casi no se mueve (entre 0.888 y 0.892). La mejor época fue la 5 y el entrenamiento se detuvo en la 16. La exactitud es una señal tosca con clases desbalanceadas, como se advirtió en la sección 2, de modo que el punto exacto de detención es algo ruidoso (la época 14 empata con la 5); lo importante es que se detiene pronto.

**Sobreajuste (sección 6).** Sin detención temprana y con `alpha` pequeño la red memoriza *train*: AUC-PR 0.950 en *train* contra 0.304 en validación (brecha de 0.646 con `alpha = 0.01`; 0.676 con `alpha ≈ 0`), es decir, un `alpha` débil **no basta por sí solo**. Con la configuración elegida la brecha es de 0.024 (0.470 contra 0.446). Un `alpha = 1` sin detención temprana también controla el sobreajuste (brecha 0.006, 38 épocas) pero con un AUC-PR de validación algo menor (0.424), diferencia del tamaño de la variación entre semillas, así que no es concluyente. Las dos defensas son complementarias: la detención temprana evita la memorización y la L2 estabiliza los pesos.

**Resultado en test.** Con el umbral de 0.23 (elegido en validación): AUC-PR 0.422, AUC-ROC 0.785, F1 0.463, precisión 0.476, sensibilidad 0.451 y exactitud 0.878. Es el mejor AUC-PR y AUC-ROC de los modelos hechos hasta ahora (SVM: 0.410 y 0.779; regresión logística: 0.381 y 0.759; Naive Bayes: 0.359 y 0.750); el F1 de la SVM (0.471) es ligeramente mayor, porque cada modelo usa un umbral distinto.

**Variabilidad (sección 7).** Entre folds, el AUC-PR fue 0.424 ± 0.018 y el AUC-ROC 0.780 ± 0.009; el resultado en *test* (0.422) coincide con esa media. Entre **semillas**, en validación el AUC-PR fue 0.433 ± 0.014 (entre 0.410 y 0.450) y el AUC-ROC 0.787 ± 0.009. Es decir, solo cambiar la inicialización de los pesos mueve el AUC-PR tanto como cambiar de fold. Dos consecuencias: (1) el modelo final depende de una semilla (42), de modo que conviene citar la media ± desviación y no un solo número; y (2) la ventaja de la red sobre la SVM en CV (0.424 contra 0.417) es menor que esta variación, por lo que **no se puede afirmar que la red sea mejor que la SVM** sin la prueba por pares del notebook de evaluación.

**Probabilidades.** A diferencia de la SVM, la red neuronal se entrena con entropía cruzada, así que su salida sí está pensada como probabilidad; aun así, con detención temprana y sin ponderar clases conviene comprobar la calibración antes de interpretarla como probabilidad real (ver el notebook de evaluación).